# BLOCK-T4 - Warmup Camera Hexapods(MTHexapod:1)

In [ ]:
import numpy as np
import os

from lsst.ts.block.utils import build_configuration_schema
from lsst.ts.observing import ObservingBlock, ObservingScript

In [ ]:
name = "BLOCK-T4"
program = "BLOCK-T4"
reason = "BLOCK-T4"
constraints = []
scripts = []

try:
    output_folder = (
        os.environ["TS_CONFIG_OCS_DIR"] + "/Scheduler/observing_blocks_maintel"
    )
except KeyError:
    warnings.warn(
        "The environment variable 'TS_CONFIG_OCS_DIR' is not set. Using default folder 'output_blocks'."
    )
    output_folder = "output_blocks"

Define the configurable properties that we will use in the configuration schema

In [ ]:
properties = {
    "step_size": {
        "description": "Step size in microns for z axis",
        "type": "number",
        "default": "100",
    },
    "max_position": {
        "description": "Absolute maximum position for z axis",
        "type": "number",
        "default": "5000",
    },
    "max_verification_position": {
        "description": "Maximum verification position for z axis",
        "type": "number",
        "default": "5000",
    },
    "max_warmup_iterations": {
        "description": "Maximum number of warmup iterations for z axis",
        "type": "number",
        "default": "5",
    },
    "sleep_time": {
        "description": "Sleep time in seconds between movements",
        "type": "number",
        "default": "2",
    },
}

block_number = name.split("-")[-1]
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

In [ ]:
# Disable hexapod compensation mode
disable_compensation = ObservingScript(
    name="maintel/disable_hexapod_compensation_mode.py",
    standard=True,
    parameters=dict(
        components=["CameraHexapod"],
    ),
)

scripts.append(disable_compensation)

# Zero position block
zero_position = ObservingScript(
    name="run_command.py",
    standard=True,
    parameters=dict(
        component="MTHexapod:1",
        cmd="move",
        parameters=dict(x=0, y=0, z=0, u=0, v=0, w=0),
    ),
)

scripts.append(zero_position)

# Warm up z axis
warmup_z = ObservingScript(
    name="maintel/warmup_hexapod.py",
    standard=False,
    parameters=dict(
        hexapod="camera",
        axis="z",
        step_size="$step_size",
        max_position="$max_position",
        max_verification_position="$max_verification_position",
        max_warmup_iterations="$max_warmup_iterations",
        sleep_time="$sleep_time",
    ),
)

scripts.append(warmup_z)
scripts.append(zero_position)

In [ ]:
block = ObservingBlock(
    name=name,
    program=program,
    configuration_schema=configuration_schema,
    scripts=scripts,
)

In [ ]:
block.model_dump_json(indent=2)

os.makedirs(output_folder, exist_ok=True)
output_path = f"{output_folder}/{name}.json"

with open(output_path, "w") as file:
    file.write(block.model_dump_json(indent=4))